# CONTINUUM: crash recovery in a notebook

An agent analysing 1,000 documents is killed at document 400 with `os._exit(9)`:
no cleanup, no flush, no chance to record where it got to. While it is down, the
dataset it depends on moves from v3 to v4. It restarts and has to work out what,
if anything, it can still trust.

Nothing here is simulated. A real child process really dies, a real side effect
is really written to disk, and every recovery decision is computed from the
durable event log by the installed library.

In order:

1. the run starts on dataset v3 and is killed at document 400
2. what survived the kill (the process is gone, the state is not)
3. the dataset moves v3 to v4, and CONTINUUM refuses to resume
4. the uncertain side effect is reconciled against the real system
5. the run finishes without repeating work or duplicating the side effect

It takes about fifteen seconds and writes only to a temporary directory, which
the last cell deletes.

## 0. Install

Colab and Binder start from an empty environment, so this installs CONTINUUM
from source. Opened from a clone that already has it, the cell does nothing.

`pip` is called through `subprocess` rather than `%pip` so that this notebook is
also plain Python: `tests/test_demo_notebook.py` executes these cells to keep
the walkthrough from rotting, and line magics would not survive that.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("continuum") is None:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "git+https://github.com/Cyrax321/CONTINUUM.git",
        ],
        check=True,
    )

## 1. The workspace

Everything the demo writes goes into a fresh temporary directory: the event log
(`agent.db`), the worker it spawns, and `github-issues.log`, which stands in for
an external system that has really been written to. A clone's `demo-run/` holds
tracked files, so the notebook stays out of it.

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

import continuum
from continuum import (
    ActionLedger,
    CheckpointManager,
    EventType,
    ProbeReconciler,
    RecoveryEngine,
    Resolution,
    SemanticPolicy,
    SQLiteStorage,
    StaticProvider,
    capture_environment,
    project,
    reconcile_pending,
)
from continuum.cli.exitcodes import exit_code_for

print("CONTINUUM", continuum.__version__, "on Python", sys.version.split()[0])

workspace = Path(tempfile.mkdtemp(prefix="continuum-demo-"))
db = workspace / "agent.db"
effects = workspace / "github-issues.log"
print("workspace", workspace)

## 2. The agent starts on dataset v3, and dies at document 400

The worker runs as a real child process because it ends in `os._exit(9)`, which
would take this kernel down with it. It is the same worker as
`examples/crash_recovery_agent.py`, so both demos reach the same verdict.

At document 400 it claims an action in the ledger, writes the side effect (a
GitHub issue), and is killed before it can record the outcome. That gap is the
whole problem: the claim is durable, the result is not.

In [ ]:
WORKER = """
import os, sys
from continuum import (ActionLedger, CheckpointManager, EventType, Run, SQLiteStorage,
                       SemanticPolicy, StaticProvider, capture_environment)

db, effects, dataset = sys.argv[1], sys.argv[2], sys.argv[3]
store = SQLiteStorage(db)
manager = CheckpointManager(store, policy=SemanticPolicy(progress_stride=200))
env = capture_environment("run_4821", StaticProvider(dataset=dataset))

store.create_run(Run(run_id="run_4821", goal="Analyze 1,000 documents"))
store.append_event("run_4821", EventType.RUN_STARTED,
                   {"goal": "Analyze 1,000 documents", "total": 1_000})
store.append_event("run_4821", EventType.DEPENDENCY_DECLARED,
                   {"resource": "dataset", "version": dataset})
store.append_event("run_4821", EventType.EVIDENCE_ADDED,
                   {"evidence_id": "paper_128", "summary": "peer-reviewed study",
                    "source": "dataset"})
store.append_event("run_4821", EventType.FINDING_ADDED,
                   {"finding_id": "finding_17", "claim": "X holds",
                    "evidence": ["paper_128"], "confidence": 0.91})

for i in range(1_000):
    store.append_event("run_4821", EventType.WORK_COMPLETED, {"doc": i})
    manager.maybe_checkpoint("run_4821", environment=env)

    if i == 399:
        ledger = ActionLedger(store, "run_4821")
        if ledger.claim("github.create_issue", {"title": "Anomaly in batch 7"}).fresh:
            with open(effects, "a") as fh:          # the real external side effect
                fh.write("issue #481: Anomaly in batch 7\\n")
            print("created GitHub issue #481", flush=True)
            print("PROCESS TERMINATED", flush=True)
            sys.stdout.flush()   # os._exit skips stdio teardown
            os._exit(9)          # no cleanup, no atexit, no close
"""
worker = workspace / "worker.py"
worker.write_text(WORKER)

# The child has to import the same CONTINUUM this kernel imported, whether that
# is site-packages (Colab, Binder) or src/ (a clone installed with -e).
child_env = dict(os.environ)
child_env["PYTHONPATH"] = os.pathsep.join(
    path
    for path in (str(Path(continuum.__file__).resolve().parents[1]), child_env.get("PYTHONPATH"))
    if path
)

first = subprocess.run(
    [sys.executable, str(worker), str(db), str(effects), "v3"],
    env=child_env,
    capture_output=True,
    text=True,
)
print(first.stdout.rstrip() or first.stderr.rstrip())
print(f"exit code {first.returncode}: killed at document 400, no cleanup, no flush")

## 3. What survived

The process is gone. The state is not. `project()` replays the event log into a
semantic state, and `verify_events()` recomputes the hash chain over it, so a
reader can tell an intact log from a tampered or truncated one.

In [ ]:
store = SQLiteStorage(str(db))
state = project("run_4821", store.read_events("run_4821"))

print("goal                ", state.goal.description if state.goal else "(none)")
print("progress            ", f"{state.progress.completed}/1000 documents")
print("findings            ", len(state.findings))
print("evidence            ", len(state.evidence))
print("event chain verified", store.verify_events("run_4821").ok)
print("side effect on disk ", effects.read_text().strip() or "(nothing)")

## 4. The dataset moves v3 to v4

`RecoveryEngine.assess()` compares the checkpoint against the environment it is
handed. Two things are wrong now: the dependency the run declared moved under it,
and one action has an unknown outcome.

Watch the mode come back `REQUEST_HUMAN`, and watch the exit code with it.
`continuum resume` maps that mode to 20, so the shell idiom

```bash
continuum resume "$RUN" --env dataset=v4 && ./start-agent.sh
```

short-circuits, and the agent is never launched onto state nobody has verified.

In [ ]:
current = capture_environment("run_4821", StaticProvider(dataset="v4"))
decision = RecoveryEngine(store, strict_unknown=True).assess(
    "run_4821", current_environment=current
)

print(decision.render())
print()
print("mode                ", decision.mode.value)
print("safe to resume      ", decision.safe)
print("next allowed action ", decision.next_allowed_action)
print("continuum resume    ", f"would exit {exit_code_for(decision.mode)}")

## 5. Reconcile the uncertain side effect

Did the GitHub issue actually get created? The ledger cannot know, so it refuses
to guess: an unknown outcome stays unknown until something asks the real system.

`ProbeReconciler` is that something. Here it answers from a lambda; in
production it would call the API and return what it found, or
`.continuum/reconcilers.json` would name a command to run.

In [ ]:
ledger = ActionLedger(store, "run_4821")
print("unresolved before ", len(ledger.pending()))

report = reconcile_pending(
    ledger,
    ProbeReconciler(lambda action: Resolution(occurred=True, external_id="481")),
)
print("probe says        ", report.render())
print("unresolved after  ", len(ledger.pending()))

## 6. Finish the job

The restored checkpoint says 400 documents are done, so the run picks up at 400
rather than from zero. Claiming the same action a second time returns
`fresh=False` and hands back the external id the probe found, which is what keeps
the second attempt from filing a duplicate issue.

In [ ]:
manager = CheckpointManager(store, policy=SemanticPolicy(progress_stride=200))
env = capture_environment("run_4821", StaticProvider(dataset="v3"))
done = manager.restore("run_4821").state.progress.completed
print(f"resumed at {done} documents, nothing reprocessed")

again = ActionLedger(store, "run_4821").claim(
    "github.create_issue", {"title": "Anomaly in batch 7"}
)
print(f"claimed again: fresh={again.fresh}, external_id={again.external_id!r}")

for doc in range(done, 1_000):
    store.append_event("run_4821", EventType.WORK_COMPLETED, {"doc": doc})
    manager.maybe_checkpoint("run_4821", environment=env)
manager.checkpoint("run_4821", environment=env)
print(f"finished at {manager.restore('run_4821').state.progress.completed} documents")

## 7. Did we do anything twice?

The five numbers `examples/crash_recovery_agent.py` prints, from the same event
log. A clean run ends with `No work repeated. No side effect duplicated.`

In [ ]:
docs = [
    event.payload["doc"]
    for event in store.read_events("run_4821")
    if event.type is EventType.WORK_COMPLETED
]
issues = [line for line in effects.read_text().splitlines() if line.strip()]
final = project("run_4821", store.read_events("run_4821"))

rows = [
    ("documents processed", str(len(docs))),
    ("duplicates", str(len(docs) - len(set(docs)))),
    ("GitHub issues created", str(len(issues))),
    ("progress recovered", f"{final.progress.completed}/1000"),
    ("event chain verified", str(store.verify_events("run_4821").ok)),
]
for label, value in rows:
    print(f"{label:<24} {value}")

ok = len(docs) == len(set(docs)) and len(issues) == 1
print()
print("No work repeated. No side effect duplicated." if ok else "SOMETHING WAS DUPLICATED")

## What to try next

Skip the cleanup cell below to keep the event log, then read it with the CLI
(`--db` comes before the subcommand, and the path is printed by cell 1):

```bash
continuum --db /tmp/continuum-demo-xxxx/agent.db inspect run_4821
continuum --db /tmp/continuum-demo-xxxx/agent.db history run_4821
continuum --db /tmp/continuum-demo-xxxx/agent.db actions run_4821
continuum --db /tmp/continuum-demo-xxxx/agent.db verify  run_4821
```

- `examples/crash_recovery_agent.py` is this walkthrough as one command, and
  `docker run --rm ghcr.io/cyrax321/continuum` runs it without a clone.
- `docs/recovery_walkthrough.md` follows one failure from adapter error to sealed
  recovery contract.
- `references/install.md` has the extras matrix, `docs/api/cli.md` the full CLI.

In [ ]:
shutil.rmtree(workspace, ignore_errors=True)
print("removed", workspace)